# 🤖 AI Resume Screening System using LangChain + Groq

## 🎯 Objective
Build an AI-powered resume screening system that evaluates candidates based on a job description.

## 🔁 Pipeline Architecture
Resume → Skill Extraction → Matching → Scoring → Explanation

## 🛠 Tools & Technologies
- Python
- LangChain
- Groq API (LLM)
- LangSmith (Tracing)

---

## 📌 Problem Statement
Develop an AI tool for recruiters that:
- Takes Resume + Job Description
- Extracts key information
- Matches candidate with requirements
- Assigns score (0–100)
- Explains reasoning

In [1]:
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os
import json

## 🔑 Step 1: Load Environment Variables & Enable LangSmith

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

print("Groq Key Loaded:", bool(os.getenv("GROQ_API_KEY")))

# LangSmith
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Resume Screening System"

Groq Key Loaded: True


## 🤖 Step 2: Initialize Groq LLM

In [3]:
from langchain_groq import ChatGroq

def get_llm(temp=0.1):
    return ChatGroq(
        model="llama-3.1-8b-instant",
        temperature=temp
    )

llm = get_llm()
print("✅ Groq LLM Ready")

✅ Groq LLM Ready


## 📄 Step 3: Define Job Description & Sample Resumes

We define realistic and detailed resumes for:
- Strong Candidate
- Average Candidate
- Weak Candidate

In [4]:
JOB_DESCRIPTION = """
Role: Senior Data Scientist

Company Overview:
A fast-growing AI-driven organization seeking a highly skilled Data Scientist to work on cutting-edge machine learning solutions.

Key Responsibilities:
- Develop, train, and deploy machine learning models
- Perform data preprocessing, feature engineering, and model optimization
- Work with structured and unstructured data
- Collaborate with cross-functional teams

Required Skills:
- Strong proficiency in Python
- Expertise in Machine Learning & Deep Learning
- Experience with NLP techniques
- Strong SQL and data manipulation skills

Tools & Technologies:
- Pandas, NumPy, Scikit-learn
- TensorFlow / PyTorch
- Docker, MLflow
- Git version control

Experience:
- Minimum 3+ years in Data Science or related field

Education:
- Bachelor’s or Master’s in Computer Science / Data Science
"""

RESUME_STRONG = """
Candidate Name: Rahul Sharma

Experience:
5 years of professional experience in Data Science and Machine Learning.

Skills:
Python, Machine Learning, Deep Learning, Natural Language Processing, SQL

Tools:
Pandas, NumPy, Scikit-learn, TensorFlow, PyTorch, Docker, MLflow, Git

Projects:
- Built NLP chatbot using Transformers
- Developed fraud detection ML models
- Optimized recommendation systems

Education:
Master’s in Data Science

Summary:
Highly experienced data scientist with strong problem-solving skills and hands-on experience in deploying scalable ML systems.
"""

RESUME_AVERAGE = """
Candidate Name: Priya Verma

Experience:
2 years of experience in Python development and basic machine learning.

Skills:
Python, basic Machine Learning, SQL

Tools:
Pandas, NumPy, Scikit-learn

Projects:
- Worked on data cleaning and analysis tasks
- Built basic classification models

Education:
Bachelor’s in Computer Science

Summary:
Motivated individual with foundational knowledge in data science but limited real-world deployment experience.
"""

RESUME_WEAK = """
Candidate Name: Arjun Kumar

Experience:
Fresher

Skills:
Basic computer knowledge, MS Excel

Tools:
None

Projects:
None

Education:
Bachelor’s in Commerce

Summary:
Entry-level candidate with no relevant experience in data science or machine learning.
"""

## 🧠 Step 4: Prompt Engineering

We define structured prompts for:
- Extraction
- Matching
- Scoring
- Explanation

## Extraction Prompt

In [5]:
extraction_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
You are an expert resume parser.

Your task is to extract ONLY the information explicitly present in the resume.

Resume:
{resume}

Return STRICT JSON format:
{{
  "skills": [],
  "experience": "",
  "tools": [],
  "education": "",
  "projects": []
}}

Rules:
- Do NOT assume missing details
- Do NOT hallucinate
- Extract only what is written
- If something is missing, leave it empty
- Output ONLY valid JSON (no explanation)
"""
)

## Matching Prompt

In [6]:
matching_prompt = PromptTemplate(
    input_variables=["profile", "job"],
    template="""
You are a professional recruiter.

Compare the candidate profile with the job description.

Job Description:
{job}

Candidate Profile:
{profile}

Evaluate based on:
- Skills match
- Experience relevance
- Tools familiarity
- Education background

Return JSON:
{{
  "skills_match": "strong/partial/weak",
  "experience_match": "strong/partial/weak",
  "tools_match": "strong/partial/weak",
  "education_match": "strong/partial/weak",
  "overall_match": "strong/partial/weak"
}}

Rules:
- Be strict and realistic
- Do NOT assume missing skills
- Base evaluation ONLY on given data
- Output ONLY JSON
"""
)

## Scoring Prompt

In [7]:
from langchain_core.prompts import PromptTemplate

scoring_prompt = PromptTemplate(
    input_variables=["profile", "match", "job"],
    template="""
You are a STRICT AI hiring evaluator.

You MUST return ONLY valid JSON.

DO NOT include:
- explanations outside JSON
- markdown
- extra text

Follow this EXACT structure:

{{
  "skills_score": number,
  "experience_score": number,
  "education_score": number,
  "tools_score": number,
  "total_score": number,
  "recommendation": "Strong Hire | Consider | Reject",
  "reason": "short explanation"
}}

--------------------------------

Scoring rules:

- Skills match → up to 40 points
- Experience → up to 25 points
- Education → up to 15 points
- Tools → up to 20 points

Total MUST equal sum of all scores (out of 100).

--------------------------------

Decision rules (VERY IMPORTANT):

- If total_score >= 70 → recommendation = "Strong Hire"
- If total_score between 50 and 69 → recommendation = "Consider"
- If total_score < 50 → recommendation = "Reject"

--------------------------------

Strict instructions:

- Evaluate ONLY based on given data
- Do NOT assume missing skills
- Penalize missing experience heavily
- Ensure scores reflect match analysis

--------------------------------

Job Description:
{job}

Candidate Profile:
{profile}

Match Analysis:
{match}

--------------------------------

Return ONLY JSON.
"""
)

## Explain Prompt

In [8]:
explain_prompt = PromptTemplate(
    input_variables=["score", "match"],
    template="""
You are a recruiter explaining a hiring decision.

Score Details:
{score}

Match Analysis:
{match}

Task:
Provide EXACTLY 2-3 lines explanation.

Include:
1. Reason for assigned score
2. Key strengths
3. Key weaknesses

Rules:
- Be concise
- Be professional
- Do NOT add extra paragraphs
- Do NOT return JSON
- Base explanation ONLY on provided data
"""
)

## 🔗 Step 5: Build LCEL Chains

In [9]:
parser = StrOutputParser()

extraction_chain = extraction_prompt | llm | parser
matching_chain = matching_prompt | llm | parser
scoring_chain = scoring_prompt | llm | parser
explain_chain = explain_prompt | llm | parser

## 🛠 Step 6: Helper Function

In [10]:
def parse_json(text):
    import json
    import re
    try:
        return json.loads(text)
    except:
        # Extract JSON from messy output
        match = re.search(r'\{[\s\S]*\}', text)
        if match:
            try:
                return json.loads(match.group())
            except:
                pass
        print("⚠️ Failed parsing:", text)
        return {"error": text}

## 🚀 Step 7: Main Pipeline using `.invoke()`

In [11]:
def screen_resume(resume, job_description, candidate_type):

    print(f"\n🔍 Screening: {candidate_type}")

    # -------------------------------
    # STEP 1: EXTRACTION
    # -------------------------------
    ext_text = extraction_chain.invoke(
        {"resume": resume},
        config={"tags": ["extraction", candidate_type]}
    )

    profile = parse_json(ext_text)
    print("\n📄 Extracted Profile:", profile)


    # -------------------------------
    # STEP 2: MATCHING
    # -------------------------------
    match_text = matching_chain.invoke(
        {
            "profile": json.dumps(profile),
            "job": job_description
        },
        config={"tags": ["matching", candidate_type]}
    )

    match_data = parse_json(match_text)
    print("\n🎯 Match Analysis:", match_data)


    # -------------------------------
    # STEP 3: SCORING
    # -------------------------------
    score_text = scoring_chain.invoke(
        {
            "profile": json.dumps(profile),
            "match": json.dumps(match_data),
            "job": job_description
        },
        config={"tags": ["scoring", candidate_type]}
    )

    score_data = parse_json(score_text)
    print("\n📊 Score:", score_data)


    # -------------------------------
    # STEP 4: EXPLANATION
    # -------------------------------
    explanation_text = explain_chain.invoke(
        {
            "score": json.dumps(score_data),
            "match": json.dumps(match_data)
        },
        config={"tags": ["explanation", candidate_type]}
    )

    print("\n🧠 Explanation:", explanation_text)


    # -------------------------------
    # RETURN FINAL OUTPUT
    # -------------------------------
    return {
        "label": candidate_type,
        "profile": profile,
        "match": match_data,
        "score": score_data,
        "explanation": explanation_text
    }

## 🧪 Step 8: Run Test Cases

In [12]:
r1 = screen_resume(RESUME_STRONG, JOB_DESCRIPTION, "STRONG")
r2 = screen_resume(RESUME_AVERAGE, JOB_DESCRIPTION, "AVERAGE")
r3 = screen_resume(RESUME_WEAK, JOB_DESCRIPTION, "WEAK")


🔍 Screening: STRONG

📄 Extracted Profile: {'skills': ['Python', 'Machine Learning', 'Deep Learning', 'Natural Language Processing', 'SQL'], 'experience': '5 years of professional experience in Data Science and Machine Learning.', 'tools': ['Pandas', 'NumPy', 'Scikit-learn', 'TensorFlow', 'PyTorch', 'Docker', 'MLflow', 'Git'], 'education': 'Master’s in Data Science', 'projects': ['Built NLP chatbot using Transformers', 'Developed fraud detection ML models', 'Optimized recommendation systems']}

🎯 Match Analysis: {'skills_match': 'strong', 'experience_match': 'weak', 'tools_match': 'strong', 'education_match': 'strong', 'overall_match': 'partial'}

📊 Score: {'skills_score': 35, 'experience_score': 5, 'education_score': 15, 'tools_score': 20, 'total_score': 75, 'recommendation': 'Strong Hire', 'reason': 'Strong skills and education match, but weak experience match.'}

🧠 Explanation: The candidate received a total score of 75 due to a strong skills and education match, but a weak experien

## 📊 Step 9: Results Summary

In [13]:
for r in [r1, r2, r3]:
    print(r["label"], "→", r["score"].get("total_score"))

STRONG → 75
AVERAGE → 50
WEAK → 0


## 🐞 Step 10: Debugging Edge Case

In [14]:
edge = extraction_chain.invoke({"resume": "Hardworking person"})
print(edge)

{
  "skills": [],
  "experience": "",
  "tools": [],
  "education": "",
  "projects": []
}


## 🎯 Step 11: Few-Shot Prompting (Bonus)

Few-shot prompting improves model consistency by providing examples.

The model learns:
- How to assign scores
- Expected output format
- Decision patterns

This reduces randomness and improves reliability.

In [15]:
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate
import json

# -------------------------------
# Example prompt (SAFE)
# -------------------------------
example_prompt = PromptTemplate(
    input_variables=["profile", "match", "job", "output"],
    template="""
Profile: {profile}
Match: {match}
Job: {job}
Result: {output}
"""
)

# -------------------------------
# Examples (JSON as STRING)
# -------------------------------
examples = [
    {
        "profile": "Python, Machine Learning, SQL, 5 years",
        "match": "Strong match",
        "job": "Data Scientist role",
        "output": "total_score: 90, recommendation: Strong Hire"
    },
    {
        "profile": "Python, SQL, 2 years",
        "match": "Moderate match",
        "job": "Data Scientist role",
        "output": "total_score: 60, recommendation: Consider"
    },
    {
        "profile": "Basic computer skills",
        "match": "Weak match",
        "job": "Data Scientist role",
        "output": "total_score: 20, recommendation: Reject"
    }
]

# -------------------------------
# Few-shot prompt (NO JSON HERE)
# -------------------------------
few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix="""
You are a strict AI hiring evaluator.

Rules:
- Return ONLY JSON
- No explanation
- No extra text

Output must contain:
total_score (number)
recommendation (Strong Hire / Consider / Reject)
""",
    suffix="""
Evaluate the candidate:

Profile: {profile}
Match: {match}
Job: {job}

Return ONLY JSON.
""",
    input_variables=["profile", "match", "job"]
)

# -------------------------------
# Chain
# -------------------------------
few_shot_chain = few_shot_prompt | llm

# -------------------------------
# Test
# -------------------------------
result = few_shot_chain.invoke({
    "profile": "Python, SQL, basic ML, 3 years experience",
    "match": "Moderate match",
    "job": "Data Scientist role"
})

# -------------------------------
# Clean JSON safely
# -------------------------------
text = result.content

# Extract JSON part only (SAFE)
start = text.find("{")
end = text.rfind("}") + 1
json_text = text[start:end]

clean_output = json.loads(json_text)

print("🎯 Few-shot Result:")
print(clean_output)

🎯 Few-shot Result:
{'total_score': 70, 'recommendation': 'Consider'}
